# CSE 532 Final Project
Hello and welcome to **Jackson Kyle** and **Samuel Cooper's** final project for CSE 532.


Research Question 1:
Is there a relationship between metadata about a film and the public perception of the quality of the film.
Judging if a film is good or not is a complicated answer to deliver about any given film, like most art forms, a film's quality is a subjective opinion formed by any viewer or consumer of the film itself. But there may be external factors, that influence whether or not a film will be considered good or bad. Are there external factors that influence whether or not a movie is considered worthwhile or not.
Using the movie data base (TMDB) as the main reference and comparing the average vote for each movie compared and averaged with the rating of famous movie critic Roger Ebert, the question of whether or not meta data about a film has a relationship with the public perception of the quality of a film. 
In this project, we investigate whether metadata can influence public perception of the quality of a movie. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import json

# 1.1) Data Loading:

In [2]:
# Loads in the tmdb datasets
tmdb_movies = pd.read_csv("tmdb_5000_movies.csv")
tmdb_credits = pd.read_csv("tmdb_5000_credits.csv")

# Renames the movie_id column to id for consistency and merging
tmdb_credits.rename(columns={"movie_id":"id"}, inplace=True)
tmdb = pd.merge(tmdb_movies, tmdb_credits, on="id")
# Cleans up duplicate columns created from the merge
tmdb.rename(columns={"title_x":"title"}, inplace=True)
tmdb.drop(columns=["title_y"], inplace=True)

# Loads in the rotten tomatoes datasets with the desired information
rt_movies = pd.read_csv("rotten_tomatoes_movies.csv", usecols=["rotten_tomatoes_link", "movie_title"])
rt_reviews = pd.read_csv("rotten_tomatoes_critic_reviews.csv", usecols=["rotten_tomatoes_link", "critic_name", "review_score"])

# Extracts the Ebert reviews from the dataset
ebert_reviews = rt_reviews.loc[rt_reviews['critic_name'] == "Roger Ebert"].reset_index(drop=True)
ebert_reviews = ebert_reviews.merge(rt_movies[['rotten_tomatoes_link', 'movie_title']],on='rotten_tomatoes_link',how='left')

# Creates the full movie info data frame
movie_info = pd.merge(tmdb, ebert_reviews, left_on="title", right_on="movie_title", how="inner")

# For the sake of our research, we want to add profit and ROI columns:
movie_info["profit"] = movie_info["revenue"] - movie_info["budget"]
movie_info["roi"] = np.where(movie_info["budget"] > 0, movie_info["profit"] / movie_info["budget"], np.nan)

movie_info["genres"] = movie_info["genres"].apply(json.loads)
movie_info["genre_names"] = movie_info["genres"].apply(lambda x: [g["name"] for g in x])

movie_info["production_companies"] = movie_info["production_companies"].apply(json.loads)
movie_info["production_company_names"] = movie_info["production_companies"].apply(lambda x: [p["name"] for p in x])

# 1.2) Data Cleaning

In [3]:
# Homepage and tagline columns are dropped because they don't contain needed information and have multiple null values
movie_info = movie_info.drop(columns=["homepage", "tagline"])

# Removes any movies that had a unknown ($0) budget so that there would be accurate ROI calculations
movie_info = movie_info.dropna(subset=["roi"])

'''Converts the original string rating to a number represented as a percentage'''
def enumerate_rating(score):
    try:
        numerator, denominator = score.split("/")
        return float(numerator) / float(denominator)
    except:
        return None

movie_info["review_score"] = movie_info["review_score"].apply(enumerate_rating)
# Removes all submissions that have an invalid review
movie_info = movie_info.dropna(subset=["review_score"])

# Checks to see how many null values are in each row
print(movie_info.isnull().sum())

print(len(movie_info))

budget                      0
genres                      0
id                          0
keywords                    0
original_language           0
original_title              0
overview                    0
popularity                  0
production_companies        0
production_countries        0
release_date                0
revenue                     0
runtime                     0
spoken_languages            0
status                      0
title                       0
vote_average                0
vote_count                  0
cast                        0
crew                        0
rotten_tomatoes_link        0
critic_name                 0
review_score                0
movie_title                 0
profit                      0
roi                         0
genre_names                 0
production_company_names    0
dtype: int64
2385


For cleaning our data, there were multiple choices we made so that we could make the most out of the data for our research questions. To begin, we dropped the movies that did not contain a review score, as there were less than 10 of them, and would not contribute to the overall data that much. Following this, we dropped the homepage and tagline rows, as they both contained many null values and had no correlation to the purpose of our research. Finally, we wanted the review score to be represented as a number, rather than a string, so that calculations could be done easier. To do this, we created the function enumerate_rating to parse the review_score and convert it to a float value. We then applied this function to each row of the data.

# 1.3) Descriptive Statistics

In [4]:
print("Revenue Statistics:")
print(movie_info["revenue"].describe())
print(f"var      {movie_info["revenue"].var()}\n")

print("Profit Statistics:")
print(movie_info["profit"].describe())
print(f"var      {movie_info["profit"].var()}\n")

print("ROI Statistics:")
print(movie_info["roi"].describe())
print(f"var      {movie_info["roi"].var()}\n")

print("Ebert Rating Statistics:")
print(movie_info["review_score"].describe())
print(f"var      {movie_info["review_score"].var()}\n")

print("TMDB Rating Statistics:")
print(movie_info["vote_average"].describe())
print(f"var      {movie_info["vote_average"].var()}")

Revenue Statistics:
count    2.385000e+03
mean     1.173744e+08
std      1.780491e+08
min      0.000000e+00
25%      1.383513e+07
50%      5.481930e+07
75%      1.457715e+08
max      2.787965e+09
Name: revenue, dtype: float64
var      3.1701474347497556e+16

Profit Statistics:
count    2.385000e+03
mean     7.600476e+07
std      1.535101e+08
min     -1.500000e+08
25%     -3.000000e+06
50%      2.306344e+07
75%      9.726388e+07
max      2.550965e+09
Name: profit, dtype: float64
var      2.3565350645607412e+16

ROI Statistics:
count     2385.000000
mean        10.924277
std        277.597095
min         -1.000000
25%         -0.198400
50%          1.025447
75%          3.091613
max      12889.386667
Name: roi, dtype: float64
var      77060.14736255954

Ebert Rating Statistics:
count    2385.000000
mean        0.701080
std         0.216807
min         0.000000
25%         0.500000
50%         0.750000
75%         0.875000
max         1.000000
Name: review_score, dtype: float64
var      0

For our first research question, we're defining a movie's success as it's ratings and the revenue/profit it accrued.

Revenue and Profit Statistics:
Looking at the statistics obtained from the revenues and profits of each movie, it is easy to tell that there are many outliers, and the values are spread across a large range. However, this is to be expected. From the standard deviation, we can see that there is a wide variation in success measured by revenue and profit.

Ebert and TMDB Rating Statistics :
From the statistics obtained from the Ebert and TMDB ratings, we can see that the two yield fairly similar mean values for critical success. Using these two in tandem will be strong in determining the critical success of different films. Additionally, the mean and medians of the two are similar, indicating that there are not many outliers. The standard deviation for each is also fairly low, showing that there isn't too much variation in critical success.

Next, we want to answer a number of questions related to group statistics about the metadata that correlates to a movie's success. First, does our definition of success (ratings and revenue/profit) correlate to each other:

-**Do higher-rated movies make more money? (Using TMDB and Ebert ratings)**\
-**Do higher profit, revenue, and ROI movies have higher ratings?**

In [10]:
print("Average Financial success based upon TMDB Rating:")
financial_success_by_tmdb_rating = movie_info.groupby(pd.cut(movie_info["vote_average"], bins=10), observed=False)[["revenue", "profit", "roi"]].mean()
print(financial_success_by_tmdb_rating.head(10), "\n")

print("Average Financial success based upon Ebert Rating:")
financial_success_by_ebert_rating = movie_info.groupby(pd.cut(movie_info["review_score"], bins=4), observed=False)[["revenue", "profit", "roi"]].mean()
print(financial_success_by_ebert_rating.head(4), "\n")

print("Average Critical success based upon profit:")
critical_success_by_profit = movie_info.groupby(pd.cut(movie_info["profit"], bins=5), observed=False)[["vote_average", "review_score"]].mean()
print(critical_success_by_profit.head(5), "\n")

print("Average Critical success based upon revenue:")
critical_success_by_revenue = movie_info.groupby(pd.cut(movie_info["revenue"], bins=5), observed=False)[["vote_average", "review_score"]].mean()
print(critical_success_by_revenue.head(5), "\n")

print("Average Critical success based upon ROI:")
critical_success_by_roi = movie_info.groupby(pd.cut(movie_info["roi"], bins=5), observed=False)[["vote_average", "review_score"]].mean()
print(critical_success_by_roi.head(5))

Average Financial success based upon TMDB Rating:
                    revenue        profit        roi
vote_average                                        
(2.994, 3.55]  2.038590e+07 -7.114100e+06   0.026397
(3.55, 4.1]    7.927788e+06 -4.769721e+07  -0.657761
(4.1, 4.65]    3.687814e+07 -7.624560e+06  -0.256958
(4.65, 5.2]    5.757318e+07  1.374916e+07   0.468095
(5.2, 5.75]    8.887533e+07  3.983782e+07   2.531891
(5.75, 6.3]    1.096931e+08  6.335208e+07  28.405936
(6.3, 6.85]    1.193854e+08  7.999967e+07   3.875820
(6.85, 7.4]    1.325338e+08  9.721639e+07   5.769759
(7.4, 7.95]    1.952433e+08  1.605484e+08  10.047029
(7.95, 8.5]    2.015247e+08  1.734575e+08   9.908850 

Average Financial success based upon Ebert Rating:
                     revenue        profit        roi
review_score                                         
(-0.001, 0.25]  8.836020e+07  4.765695e+07   3.875904
(0.25, 0.5]     1.005659e+08  5.553201e+07   2.343207
(0.5, 0.75]     1.206923e+08  7.674685e+07   

As we can see in the group statistics above, the factors in our definition of success seems to correlate. We can see that on average, movies with lower review scores didn't perform as well financially in either revenue or profit, and movies with high reviews tended to do better. Similarly, on average those movies with high profits and revenue scored higher critically. From this, we can see that our definition of a movie's success is consitent, and that those factors correlate to each other.Now, we want to see what other factors influence a movie's success.

-**Do higher budget movies make more money?**

In [6]:
financial_success_by_budget = movie_info.groupby(pd.cut(movie_info["budget"], bins=5), observed=False)[["revenue", "profit", "roi"]].mean()
print(financial_success_by_budget.head(5))

                                 revenue        profit        roi
budget                                                           
(-379998.999, 76000000.8]   7.745445e+07  4.993857e+07  12.533367
(76000000.8, 152000000.6]   2.908477e+08  1.843949e+08   1.670038
(152000000.6, 228000000.4]  5.790697e+08  3.937436e+08   2.082797
(228000000.4, 304000000.2]  1.037690e+09  7.829753e+08   3.183776
(304000000.2, 380000000.0]  1.045714e+09  6.657138e+08   1.751878


From this, we can see that on average, movies that had higher budgets resulted in higher revenues. However, this is not always the case in terms of profit, as the bin (228000000.0, 304000000.0] had a higher average profits than the higher budget bin. These values are similar however, showing that budget does have influence on both the revenue and profit of a movie.

-**Do higher budget movies have better ratings?**

In [7]:
critical_success_by_budget = movie_info.groupby(pd.cut(movie_info["budget"], bins=5), observed=False)[["vote_average", "review_score"]].mean()
print(critical_success_by_budget.head(5))

                            vote_average  review_score
budget                                                
(-379998.999, 76000000.8]       6.409463      0.701368
(76000000.8, 152000000.6]       6.314238      0.695364
(152000000.6, 228000000.4]      6.369565      0.736413
(228000000.4, 304000000.2]      6.500000      0.660714
(304000000.2, 380000000.0]      6.400000      0.500000


From this information, we can see that for the most part, each budget range has fairly similar review averages for the TMDB reviews. However, for the Ebert reviews, we can actually see that on average, the lower budget movies scored higher than higher budget movies. This leads us to believe that budget does not correlate as much to critical success. This leads us to believe that even though a movie may have a high budget and accrue a high revenue and profit, it still may perform poorly critically.

-**Do different genres correspond to more success?**

In [8]:
movie_genres = movie_info.explode("genre_names")
financial_success_by_genre = movie_genres.groupby("genre_names")[["revenue", "profit", "roi"]].mean()
print("Average Financial success by genre:")
print(financial_success_by_genre, "\n")

print("Average Critical success by genre:")
critical_success_by_genre = movie_genres.groupby("genre_names")[["vote_average", "review_score"]].mean()
print(critical_success_by_genre)

Average Financial success by genre:
                      revenue        profit        roi
genre_names                                           
Action           1.768938e+08  1.125978e+08   2.019840
Adventure        2.316112e+08  1.567217e+08   2.659200
Animation        2.827740e+08  2.010300e+08   4.092986
Comedy           9.896164e+07  6.293258e+07   3.603287
Crime            9.142737e+07  5.494205e+07   2.459739
Documentary      1.719610e+07  1.170157e+07  24.742282
Drama            8.062442e+07  4.909940e+07   3.508370
Family           1.980766e+08  1.362062e+08   3.042994
Fantasy          2.378200e+08  1.644772e+08   2.959771
Foreign          7.000000e+00  0.000000e+00   0.000000
History          1.004758e+08  5.501771e+07   2.017394
Horror           7.276727e+07  4.904267e+07  98.262282
Music            7.045968e+07  4.240393e+07   2.635151
Mystery          1.235278e+08  7.999542e+07  83.887926
Romance          9.500826e+07  6.383469e+07   3.170738
Science Fiction  1.708392e+08

From these tests we can see that some genres perform better than others both critically and financially. For example, we can see that animated movies seem to perform very well financially with high average revenue and profit values. On the other hand, we can see that documentaries tend to score very highly on average. However, analyzing this, we can see that documentaries also perform worse financially, even though they are highly rated. These findings could potentially be useful for predictions of a movie's success.

In [9]:
##########TEST#############
production_companies = movie_info.explode("production_company_names")
financial_success_by_prod_company = production_companies.groupby("production_company_names")[["revenue", "profit"]].mean()
print("Financial success by production company:")
print(financial_success_by_prod_company, "\n")

print("Critical success by production company:")
critical_success_prod_company = production_companies.groupby("production_company_names")[["vote_average", "review_score"]].mean()
print(critical_success_prod_company)

Financial success by production company:
                                      revenue        profit
production_company_names                                   
"DIA" Productions GmbH & Co. KG  4.435093e+07  8.350926e+06
10th Hole Productions            3.470585e+07  3.120585e+07
1492 Pictures                    2.847344e+08  2.066094e+08
16 Block Productions             6.566472e+07  1.066472e+07
1818                             1.223263e+07  7.232628e+06
...                                       ...           ...
Zupnik Cinema Group II           1.072523e+07 -1.774772e+06
d-rights                         2.347105e+08  2.107105e+08
double A Films                   9.304609e+06  5.304609e+06
icon                             3.031801e+06 -1.330097e+06
thinkfilm                        5.845347e+08  4.045347e+08

[2404 rows x 2 columns] 

Critical success by production company:
                                 vote_average  review_score
production_company_names                            

# 1.4) Data Visualization